# Lab 5b: Workflow de Agentes Sequencial (15 min)

## Objetivos
- Criar múltiplos agentes especializados no Foundry
- Encadear agentes numa pipeline sequencial
- Passar output de um agente como input do seguinte
- Entender o padrão Agent → Agent → Agent

## Conceitos

### Workflow de Agentes vs Workflow LLM

No Lab 5 usámos chamadas directas ao modelo. Aqui usamos **agentes Foundry** que têm:
- **Estado persistente** (threads)
- **Ferramentas** (code interpreter, functions)
- **Instruções especializadas**

### Padrão Sequential Agent Pipeline
```
┌──────────────┐    output    ┌──────────────┐    output    ┌──────────────┐
│  Agente A    │ ──────────▶ │  Agente B    │ ──────────▶ │  Agente C    │
│  (Pesquisa)  │             │ (Análise)    │             │ (Relatório)  │
└──────────────┘             └──────────────┘             └──────────────┘
```

Cada agente é um **especialista** que contribui para o resultado final.

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeInterpreterTool
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

project = AIProjectClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

print(f"✅ Projeto conectado: {endpoint}")

In [ ]:
# Helper: executar um agente e obter a resposta
def executar_agente(agent_id: str, mensagem: str) -> str:
    """Cria um thread, envia mensagem, executa e retorna a resposta."""
    thread = project.agents.threads.create()
    project.agents.messages.create(
        thread_id=thread.id,
        role="user",
        content=mensagem,
    )
    run = project.agents.runs.create_and_process(
        thread_id=thread.id,
        agent_id=agent_id,
    )
    if run.status != "completed":
        return f"❌ Erro: run status = {run.status}"

    messages = project.agents.messages.list(thread_id=thread.id)
    for msg in messages.data:
        if msg.role == "assistant":
            for item in msg.content:
                if hasattr(item, "text"):
                    return item.text.value
    return "(sem resposta)"

print("✅ Helper executar_agente() definido")

## 🖥️ Exercício 5b.1: Pipeline de 3 Agentes

Cenário: Processar um pedido de proposta técnica

1. **Agente Investigador** → Analisa requisitos e identifica tecnologias
2. **Agente Arquiteto** → Desenha a solução técnica
3. **Agente Redator** → Cria a proposta final formatada

In [ ]:
# Exercício 5b.1: Criar os 3 agentes especializados

# Agente 1: Investigador
investigador = project.agents.create_agent(
    model=model,
    name="investigador",
    instructions="""És um analista técnico. A tua tarefa é:
1. Analisar requisitos de um projeto
2. Identificar tecnologias Azure relevantes
3. Listar riscos e considerações

Responde em formato estruturado com secções claras. Português de Portugal.""",
)
print(f"✅ Agente Investigador: {investigador.id}")

# Agente 2: Arquiteto
arquiteto = project.agents.create_agent(
    model=model,
    name="arquiteto",
    instructions="""És um arquiteto de soluções Azure. A tua tarefa é:
1. Receber uma análise técnica e requisitos
2. Desenhar uma arquitetura Azure (serviços, integrações, fluxos)
3. Estimar dimensionamento e custos aproximados

Usa diagramas ASCII quando possível. Português de Portugal.""",
)
print(f"✅ Agente Arquiteto: {arquiteto.id}")

# Agente 3: Redator
redator = project.agents.create_agent(
    model=model,
    name="redator",
    instructions="""És um redator técnico. A tua tarefa é:
1. Receber análise e arquitetura técnica
2. Criar uma proposta profissional e concisa
3. Incluir: Resumo Executivo, Solução Proposta, Cronograma, Próximos Passos

Escreve de forma profissional e persuasiva. Máximo 400 palavras. Português de Portugal.""",
)
print(f"✅ Agente Redator: {redator.id}")

In [ ]:
# Executar a pipeline sequencial

pedido = """
A empresa Contoso quer migrar o seu sistema de atendimento ao cliente para a cloud.
Atualmente usam:
- Base de dados SQL Server on-premises com 500GB de dados
- Aplicação web .NET Framework 4.8
- Chatbot básico baseado em regras

Requisitos:
- Chatbot inteligente com IA (entender perguntas em linguagem natural)
- Pesquisa nos manuais de produto (3000 documentos PDF)
- Integração com o CRM existente (Dynamics 365)
- Alta disponibilidade (99.9% SLA)
- RGPD compliance (dados na Europa)
"""

print("=" * 60)
print("📋 PEDIDO DO CLIENTE")
print("=" * 60)
print(pedido)

# Passo 1: Investigador analisa
print("\n" + "=" * 60)
print("🔍 PASSO 1: Investigador")
print("=" * 60)
analise = executar_agente(investigador.id, f"Analisa os seguintes requisitos de projeto:\n{pedido}")
print(analise)

# Passo 2: Arquiteto desenha (recebe output do investigador)
print("\n" + "=" * 60)
print("🏗️ PASSO 2: Arquiteto")
print("=" * 60)
arquitetura = executar_agente(
    arquiteto.id,
    f"Com base nesta análise técnica, desenha a arquitetura Azure:\n\n{analise}\n\nRequisitos originais:\n{pedido}",
)
print(arquitetura)

# Passo 3: Redator cria proposta (recebe output dos dois anteriores)
print("\n" + "=" * 60)
print("📝 PASSO 3: Redator")
print("=" * 60)
proposta = executar_agente(
    redator.id,
    f"Cria uma proposta técnica profissional com base em:\n\nANÁLISE:\n{analise}\n\nARQUITETURA:\n{arquitetura}",
)
print(proposta)

print("\n" + "=" * 60)
print("✅ Pipeline de 3 agentes concluída!")

## 🖥️ Exercício 5b.2: Pipeline com Code Interpreter

Vamos adicionar o **Code Interpreter** a um dos agentes para gerar dados estruturados.

In [ ]:
# Exercício 5b.2: Pipeline com Code Interpreter

# Agente Coletor: recolhe dados
coletor = project.agents.create_agent(
    model=model,
    name="coletor-dados",
    instructions="""Gera dados de exemplo sobre utilização de serviços Azure.
Cria uma tabela CSV com colunas: servico, regiao, custo_mensal_eur, utilizacao_pct, tendencia.
Gera 8 linhas de dados realistas. Responde APENAS com o CSV (com header).""",
)

# Agente Analista: analisa com código
analista = project.agents.create_agent(
    model=model,
    name="analista-custos",
    instructions="""És um analista de custos Azure.
Recebe dados CSV de utilização de serviços.
Usa o code interpreter para:
1. Calcular o custo total e médio
2. Identificar o serviço mais caro
3. Identificar serviços subutilizados (< 30% utilização)
4. Dar 3 recomendações de otimização
Responde em português.""",
    tools=CodeInterpreterTool().definitions,
)

# Executar pipeline
print("⚙️ Passo 1: Gerar dados...")
dados_csv = executar_agente(coletor.id, "Gera os dados de utilização Azure.")
print(dados_csv[:300])

print("\n⚙️ Passo 2: Analisar com Code Interpreter...")
analise_custos = executar_agente(
    analista.id,
    f"Analisa estes dados de custos Azure:\n\n{dados_csv}",
)
print(analise_custos)

print("\n✅ Pipeline com Code Interpreter concluída!")

In [ ]:
# Limpeza - apagar todos os agentes criados
for a in [investigador, arquiteto, redator, coletor, analista]:
    project.agents.delete_agent(a.id)
    print(f"🗑️ Agente {a.name} eliminado")

print("\n✅ Limpeza concluída!")

## ✅ Resumo

Neste lab aprendeste:
- A criar **múltiplos agentes especializados** no Foundry
- O padrão **sequential agent pipeline** (Agente A → B → C)
- A passar output de um agente como contexto do seguinte
- A combinar agentes com **Code Interpreter** para análise de dados

### Padrões de Workflows de Agentes
| Padrão | Descrição | Quando usar |
|--------|-----------|-------------|
| **Sequential** | A → B → C | Pipelines lineares |
| **Parallel** | A + B + C → D | Tarefas independentes |
| **Router** | Input → decide qual agente usar | Classificação + ação |
| **Consensus** | Vários agentes votam | Decisões críticas |

**Próximo:** [Lab 6 - APIM](lab06-apim.ipynb)